In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn import tree
from sklearn.tree import export_text

In [ ]:
# import data
from foundry.transforms import Dataset

df_trained = Dataset.get("df_trained").read_table(format="pandas")
q_e_df =  Dataset.get("q_e_df").read_table(format="pandas")

In [ ]:
for cluster in range(df_trained["CLUSTER"].nunique()):
    print("state ", cluster)
    df_cluster = df_trained[df_trained["CLUSTER"]==cluster]
    print(df_cluster["TIME"].describe())
    plt.figure()
    plt.hist(df_cluster["TIME"], range=[0,2000], bins = 100)
    plt.xlabel("Time period")
    plt.ylabel("Frequency")
    plt.title("Time period distribution for state " + str(cluster))
    plt.show()
    print("-------------------------------------------------------------------")

# cluster heatmap

In [ ]:
df_trained["CLUSTER"] = df_trained["CLUSTER"].astype("int")

In [ ]:
def cluster_feature_heat_map_transpose(df_trained, feature_list):
    clusters = df_trained['CLUSTER'].unique()
    clusters.sort
    num_clusters = len(clusters)
    num_features = len(feature_list)
    heat_map_matrix = pd.DataFrame(index=feature_list,columns=range(num_clusters))
    for cluster in range(num_clusters):
        cluster_df = df_trained[df_trained['CLUSTER']==cluster]
        for feature in feature_list:
            feature_mean = df_trained[feature].mean()
            feature_std = df_trained[feature].std()
            heat_map_matrix.at[feature, cluster] = (cluster_df[feature].mean()-feature_mean)/feature_std
    clean_matrix = heat_map_matrix.apply(pd.to_numeric, errors='coerce')
    print(clean_matrix)
    sns.heatmap(clean_matrix,xticklabels=True, yticklabels=True, cmap = 'seismic', vmin=-2.5, vmax=2.5)
    heat_map_matrix = heat_map_matrix.astype(float)
    print(heat_map_matrix)
    sns.heatmap(heat_map_matrix,xticklabels=True, yticklabels=True, cmap = 'seismic', vmin=-2.5, vmax=2.5)

In [ ]:
# predict_cluster() takes in a clustered dataframe df_new, the number of
# features pfeatures, and returns a prediction model m that predicts the most
# likely cluster from a datapoint's features
def predict_cluster(
    df_new, pfeatures  # dataframe: trained clusters
):  # int: # of features
    # the first 2 columns are ID and TIME so column 2 is the first feature
    # this would be safer if we took in a list of feature names and directly called the columns by their names
    X = df_new.iloc[:, 2 : 2 + pfeatures]
    y = df_new["CLUSTER"]

    params = {"max_depth": [4]}

    m = DecisionTreeClassifier()
    # m = RandomForestClassifier()

    m = GridSearchCV(estimator=m, param_grid=params, cv=5)
    # m = GridSearchCV(m, params, cv = 5, iid = True) # will return warning if 'iid' param not set to true

    # m = DecisionTreeClassifier(max_depth = 10)
    m.fit(X, y)
    return m

In [ ]:
def plot_short_tree(df_trained, class_names=None):
    num_clusters = df_trained["NEXT_CLUSTER"].max()+1
    p = df_trained.shape[1] - 6 - num_clusters -1
    feature_column_list = df_trained.columns[2:2+p]
    print(feature_column_list)
    short_tree_grid = predict_cluster(df_trained,p)
    print(short_tree_grid.best_params_)
    short_tree = short_tree_grid.best_estimator_
    plt.figure(figsize=(35,10))
    tree.plot_tree(short_tree, fontsize=12, feature_names=feature_column_list,
                        class_names= class_names,
                        impurity = True,
                        proportion = False,
                        filled=True)
    plt.show()
    tree_rules = export_text(short_tree, feature_names=list(feature_column_list))
    print(tree_rules)
    
    return short_tree_grid

In [ ]:
df_trained = df_trained.rename(columns={"systolic_bp":"Systolic BP",
"diastolic_bp":"Diastolic BP",
"pulse":"Heart rate (pulse)",
"resp":"Respiratory rate",
"temp":"Temperature",
"weight":"Weight",
"fio2":"FiO2",
"spo2":"SpO2",
"sao2":"SaO2",
"glucose":"Glucose",
"magnesium":"Magnesium",
"calcium":"Calcium",
"sodium":"Sodium",
"chloride":"Chloride",
"hco3":"HCO3",
"phosphate":"Phosphate",
"potassium":"Potassium",
"bun":"BUN",
"creatinine":"Creatinine",
"ck":"CK",
"lactate":"Lactate",
"hematocrit":"Hematocrit",
"hemoglobin":"Hemoglobin",
"wbc":"WBC",
"platelet_count":"Platelets",
"ldh":"LDH",
"crp":"CRP",
"esr":"ESR",
"d_dimer":"D-Dimer",
"inr":"INR",
"pt":"PT",
"ptt":"PTT",
"high_sensitivity_crp":"hsCRP",
"high_sensitivty_troponin":"hsTroponin",
"total_bilirubin":"Total Bilirubin",
"troponin":"Troponin",
"conjugated_bilirubin":"Conjugated Bilirubin",
"alt":"ALT",
"ast":"AST",
"albumin":"Albumin",
"alkaline_phosphate":"Alkaline Phosphatase",
"bnp":"BNP",
"paco2":"PaCO2",
"pao2":"PaO2",
"ph":"pH",
"total_sofa":"Total SOFA",
"cardio_sofa":"Cardiovascular SOFA",
"coag_sofa":"Coagulation SOFA",
"liver_sofa":"Liver SOFA",
"renal_sofa_new_renal":"Renal SOFA",
"resp_sofa":"Respiratory SOFA",
"cns_sofa":"CNS SOFA",
"gcs":"GCS",
"emesis":"Emesis output",
"stool":"Stool output",
"urine_6hr_sum":"Urine output - past 6hrs",
"amt_left":"Fluid - ordered, not yet administered",
"fluid_given_this_block":"Fluid - given this hour",
"cumulative_fluid_past_6h":"Fluid - cumulative given past 6 hours"})

In [ ]:
feature_list = ["MAP",
"Systolic BP",
"Diastolic BP",
"Heart rate (pulse)",
"Respiratory rate",
"Temperature",
"Weight",
"FiO2",
"SpO2",
"SaO2",
"Glucose",
"Magnesium",
"Calcium",
"Sodium",
"Chloride",
"HCO3",
"Phosphate",
"Potassium",
"BUN",
"Creatinine",
"CK",
"Lactate",
"Hematocrit",
"Hemoglobin",
"WBC",
"Platelets",
"LDH",
"CRP",
"ESR",
"D-Dimer",
"INR",
"PT",
"PTT",
"hsCRP",
"hsTroponin",
"Total Bilirubin",
"Troponin",
"Conjugated Bilirubin",
"ALT",
"AST",
"Albumin",
"Alkaline Phosphatase",
"BNP",
"PaCO2",
"PaO2",
"pH",
"Total SOFA",
"Cardiovascular SOFA",
"Coagulation SOFA",
"Liver SOFA",
"Renal SOFA",
"Respiratory SOFA",
"CNS SOFA",
"GCS",
"Emesis output",
"Stool output",
"Urine output - past 6hrs",
"Fluid - ordered, not yet administered",
"Fluid - given this hour",
"Fluid - cumulative given past 6 hours"]

In [ ]:
for feature in feature_list:
    if feature not in df_trained.columns:
        print(feature)

In [ ]:
plt.figure(figsize=(8, 9))
cluster_feature_heat_map_transpose(df_trained, feature_list)

In [ ]:
metamodel = plot_short_tree(df_trained)

In [ ]:
pd.DataFrame(metamodel.cv_results_)

# action values and frequencies

In [ ]:
action_names = ["no fluids", "0-10 cc/kg", "10-20 cc/kg", "20-30 cc/kg", "30+ cc/kg"]

In [ ]:
def plot_action_values(actions, action_names, Q_star, mu = 0.75):
    # size of Q_star should be number of clusters x number of actions
    for index, row in Q_star.iterrows():
        exploit = mu * row
        # explore = (1 - mu) * pE

        
        plt.figure(figsize=(9,4))
        plt.bar(actions, exploit, color='#732f6e')#, label=r'$\mu\,Q^*(s,a)$')
        #plt.bar(actions, explore, bottom=exploit, color='#d481ce')#, label=r'$(1-\mu)\,\mathrm{pE}(s,a)$')
        
        # # if we want the green box that highlights an action?
        a_star = np.argmin(exploit)
        ax = plt.gca()
        ax.add_patch(plt.Rectangle((a_star-0.5, 0.1),
                                   1,
                                   (exploit).min()*1.2,
                                   fill=False,
                                   linewidth=3,
                                   edgecolor='green'))
        
        plt.xlabel("Action")
        plt.ylabel("Risk")
        plt.title("Long-term risk for taking each action in state " + str(index))
        plt.xticks(actions, action_names, rotation=45)
        #plt.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
plot_action_values(np.arange(0,5), action_names, q_e_df, mu = 1)

In [ ]:
action_freq = df_trained['ACTION'].value_counts(normalize=True)
action_freq

In [ ]:
def plot_freqs(actions, action_names, pE):
    """plots frequency of actions in each cluster (state)"""
    for cluster in range(10):
        action_counts = df_trained[df_trained['CLUSTER']==cluster]['ACTION'].value_counts(normalize=True)
        action_counts = pE.reindex(np.arange(0,5), fill_value=0)
        
        plt.figure(figsize=(9,4))
        plt.bar(actions, action_counts, color='#d481ce')
        plt.xlabel("Action")
        plt.ylabel("Frequency")
        plt.title("Frequency of actions for state " + str(cluster))
        plt.xticks(actions, action_names, rotation=45)
        #plt.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
plot_freqs(np.arange(0,5), action_names, action_freq, mu = 0)

# absorbing probabilities

p_df = Dataset.get("/path/to/p_df").read_table(format="pandas")
pi_e_df = Dataset.get("/path/to/pi_e_df").read_table(format="pandas")

In [ ]:
def build_P_pi(P_df, pi, absorbing_states=None):
    absorbing_states = set(absorbing_states) if absorbing_states is not None else set()

    n_states = int(max(P_df["CLUSTER"].max(), P_df["NEXT_CLUSTER"].max())) + 1

    pi = np.asarray(pi)
    chosen = pd.DataFrame({
        "CLUSTER": np.arange(len(pi)),
        "ACTION": pi
    })

    df = P_df.merge(chosen, on=["CLUSTER", "ACTION"], how="inner")

    P_pi = np.zeros((n_states, n_states), dtype=float)

    for s, sp, p in zip(
        df["CLUSTER"].to_numpy(),
        df["NEXT_CLUSTER"].to_numpy(),
        df["prob"].to_numpy()
    ):
        P_pi[int(s), int(sp)] = float(p)

    # Make the end states absorbing
    for s in absorbing_states:
        if s < n_states:
            P_pi[s, :] = 0.0
            P_pi[s, s] = 1.0

    return P_pi

In [ ]:
def absorbing_probabilities(P, tol= 1e-12):
    """
    Params
    P : (n,n) array_like
        Row-stochastic transition matrix.
    tol : float
        Numerical tolerance for detecting absorbing states.

    Returns
    B_full : (n, k) np.ndarray
        Absorption probabilities for every start state (rows),
        for each absorbing state (columns, in original-state index order).
        For absorbing start states, the corresponding row is a one-hot vector.
    absorbing_states : list[int]
        Indices of absorbing states
    transient_states : list[int]
        Indices of transient states
    """
    P = np.array(P, dtype=float)
    n = P.shape[0]
    if P.shape != (n, n):
        raise ValueError("P should be a square matrix")
    if np.any(P < -tol):
        raise ValueError("P has negative entries")
    if not np.allclose(P.sum(axis=1), 1.0, atol=1e-10):
        print(P.sum(axis=1))
        raise ValueError("rows of P must sum to 1!")

    # find absorbing and transient states
    absorbing_mask = (np.abs(np.diag(P) - 1.0) <= tol) & (np.sum(np.abs(P - np.eye(n)), axis=1) <= tol)
    absorbing_states = np.where(absorbing_mask)[0].tolist()
    transient_states = np.where(~absorbing_mask)[0].tolist()

    k = len(absorbing_states)
    t = len(transient_states)

    if k == 0:
        raise ValueError("No absorbing states detected. Chain is not absorbing (or tol too strict).")
    if t == 0:
        # Everything is absorbing: absorption probs are just identity over absorbing states
        B_full = np.eye(n)[:, absorbing_states]
        return B_full, absorbing_states, transient_states

    # reorder matrix[transient, absorbing]
    order = transient_states + absorbing_states
    inv_order = np.empty(n, dtype=int)
    inv_order[order] = np.arange(n)

    P_re = P[np.ix_(order, order)]
    Q = P_re[:t, :t]
    R = P_re[:t, t:]

    # get fund. matrix and absorption probabilities
    I = np.eye(t)
    try:
        N = np.linalg.inv(I - Q)
    except np.linalg.LinAlgError:
        raise ValueError("I - Q is singular; cycle within transient states")

    B_trans = N @ R  # (t,k)
    B_full = np.zeros((n, k), dtype=float)

    for i_pos, i_state in enumerate(transient_states):
        B_full[i_state, :] = B_trans[i_pos, :]

    for j_pos, j_state in enumerate(absorbing_states):
        B_full[j_state, j_pos] = 1.0

    # B[i, j] = P(absorb in absorbing[j] | start at state i)

    return B_full, absorbing_states, transient_states

In [ ]:
def absorption_after_one_action(P_df, pi, start_state, action, absorbing_states=None):
    """
    Computes absorption probabilities if you take `action` at `start_state`,
    then follow policy `pi` afterward.

    Returns
    -------
    probs : np.ndarray, shape (k,)
        Absorption probabilities into each absorbing state.
    absorbing_states : list[int]
        Absorbing/end states corresponding to columns of probs.
    """

    absorbing_states = set(absorbing_states) if absorbing_states is not None else set()

    # Build transition matrix under optimal policy
    P_pi = build_P_pi(P_df, pi, absorbing_states=absorbing_states)

    # Absorption probabilities from every state under optimal policy
    B_pi, absorbing_states_list, transient_states = absorbing_probabilities(P_pi)

    # If start_state is already absorbing, action does not matter
    if start_state in absorbing_states:
        idx = absorbing_states_list.index(start_state)
        probs = np.zeros(len(absorbing_states_list))
        probs[idx] = 1.0
        return probs, absorbing_states_list

    # Get transition distribution for taking this specific action at start_state
    row_df = P_df[
        (P_df["CLUSTER"] == start_state)
        & (P_df["ACTION"] == action)
    ]

    if row_df.empty:
        raise ValueError(f"No transitions found for state {start_state}, action {action}")

    n_states = P_pi.shape[0]
    action_row = np.zeros(n_states)

    for sp, p in zip(row_df["NEXT_CLUSTER"], row_df["prob"]):
        action_row[int(sp)] += float(p)

    if not np.isclose(action_row.sum(), 1.0, atol=1e-10):
        print("row sum:", action_row.sum())
        raise ValueError(
            f"Transition probabilities for state {start_state}, action {action} do not sum to 1"
        )

    # One-step deviation:
    # Take action now, then use optimal-policy absorption probabilities from next states
    probs = action_row @ B_pi

    return probs, absorbing_states_list

In [ ]:
P_df = p_df.copy()
P_df["ACTION"] = P_df["ACTION"].astype(int)
P_df["CLUSTER"] = P_df["CLUSTER"].astype(int)
P_df["NEXT_CLUSTER"] = P_df["NEXT_CLUSTER"].astype(int)
pi_e = pi_e_df.to_numpy()
pi_e = np.asarray(pi_e).reshape(-1)

In [ ]:
probs, end_states = absorption_after_one_action(
    P_df,
    pi_e,
    start_state=4,
    action=0,
    absorbing_states=np.arange(8,16)
)

pd.Series(probs, index=end_states)

In [ ]:
B_df = pd.DataFrame(B_full, columns=[f"end_{s}" for s in absorbing_states])
B_df["start_cluster"] = np.arange(B_df.shape[0])
B_df = B_df.set_index("start_cluster")

B_df

In [ ]:
# edit as needed
end_states_map = {
    "end_8": "left on trach",
    "end_9": "7-day readmission",
    "end_10": "left on oxygen",
    "end_11": "short rehab",
    "end_12": "expired",
    "end_13": "home",
    "end_14": "long rehab",
    "end_15": "hospice"
}

In [ ]:
B_named = B_df.rename(columns=end_states_map)

for cluster, row in B_named.iterrows():
    plt.figure(figsize=(6, 4))
    
    row.plot(kind='bar')
    
    plt.title(f"End State Distribution for Cluster {cluster}")
    plt.xlabel("End State")
    plt.ylabel("Probability")
    plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()

In [ ]:
probs, end_states = absorption_after_one_action(
    P_df,
    pi_e,
    start_state=4,
    action=0,
    absorbing_states=np.arange(8,16)
)

pd.Series(probs, index=end_states)

In [ ]:
def plot_one_action_absorption(
    P_df,
    pi,
    start_state,
    action,
    absorbing_states,
    end_states_map=None,
    figsize=(10, 5)
):
    probs, end_states = absorption_after_one_action(
        P_df,
        pi,
        start_state=start_state,
        action=action,
        absorbing_states=absorbing_states
    )

    labels = [
        end_states_map.get(f"end_{s}", f"state_{s}") if end_states_map else f"state_{s}"
        for s in end_states
    ]

    result = pd.Series(probs, index=labels)

    ax = result.plot(kind="bar", figsize=figsize)
    ax.set_ylabel("Probability of absorption")
    ax.set_xlabel("End state")
    ax.set_title(
        f"End state distribution from cluster {start_state}\n"
        f"Action {action}"
    )
    ax.set_ylim(0, 1)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    return result

In [ ]:
result = plot_one_action_absorption(
    P_df,
    pi_e,
    start_state=4,
    action=3,
    absorbing_states=np.arange(8, 16),
    end_states_map=end_states_map
)